# System 3: $V(r) = -V_0 \exp(-r^2/R^2)$ ($s$-wave bound state, $p$-wave resonance).
Define $\lambda=2mV_0R^2/\hbar^2 = V_0R^2$ (in standard units). The first $s$-wave bound state occurs at $\lambda\gtrsim 2.6840$. The first $p$-wave state reaches threshold (moves from the continuum to the bound sector) at $\lambda\approx 12.0993$. Thus, if we want an $s$-wave bound state and a narrow $p$-wave resonance, we should tune $12.0993 - \epsilon < \lambda < 6.0497$ for some small $\epsilon$.

Unfortunately, there isn't a $p$-wave resonance much narrower than the one from System 2. This is a limit of Gaussian potentials. There's nothing confining the $p$-wave resonance except for the centrifugal barrier, and that's not enough to keep it near continuum once we approach threshold (Wigner threshold law says $\Gamma\sim k^{2l+1}$ while Re$(E)\sim k^2$, so $\Gamma/$Re$(E)\sim k\rightarrow 0$ only as $k\rightarrow 0$ -- right at threshold -- which is where Re$(E)\rightarrow 0$ anyway). To get narrower, we'll likely need another barrier (see System 4).

In [4]:
from src.dvr import DVR

import numpy as np
import matplotlib.pyplot as plt

In [5]:
# define system parameters
n = 256                          # number of DVR grid points
L = 50.0                         # system size

def attractive_gaussian(V0, R):
    def gauss(r):
        return -V0 * np.exp(-r**2 / R**2)
    return gauss

In [22]:
# find bound states and p threshold
strength = np.linspace(9.0, 15.0, 100)
R, V0 = 1.0, strength

"""
for v in V0:
    potential = attractive_gaussian(v, R)
    dvrsystem = DVR(n, L, rotation_angle=0.0, potential=potential, l=1)
    eigs, _ = dvrsystem.eig()
    mask = np.real(eigs) < 0
    if len(eigs[mask]) > 0: 
        print(
            f"V0={v:.4f}",
            eigvals[mask]
        )
"""
print("p-wave threshold is near V0 = 12.0909.")

p-wave threshold is near V0 = 12.0909.


In [23]:
# scan through different potential strengths before threshold,
# and pick out narrow resonances (validated by if they don't
# move in a phi-region around where the resonance was found
strength = np.linspace(11.0, 12.1, 150)
R = 1.0
phis = np.linspace(0.01, np.pi/4, 30)   # push past pi/6 so higher-|Im| states get exposed too

def track_pole(dvrsystem, phis, seed=None, im_lo=1e-5, im_hi=.15):
    """Follow one eigenvalue across a phi sweep """
    trajectory = []
    prev = seed
    for phi in phis:
        dvrsystem.reset_rotation_angle(phi)
        eigvals, _ = dvrsystem.closest_eig_pole()
        mask = (np.imag(eigvals) < -im_lo) & (np.imag(eigvals) > -im_hi)
        cands = eigvals[mask]
        if len(cands) == 0:
            trajectory.append((phi, None))
            continue
        if prev is None:
            # first exposure: just take the one with smallest |Im| (least contaminated)
            pick = cands[np.argmin(np.abs(np.imag(cands)))]
        else:
            pick = cands[np.argmin(np.abs(cands - prev))]
        trajectory.append((phi, pick))
        prev = pick
    return trajectory

for v in strength:
    potential = attractive_gaussian(v, R)
    dvrsystem = DVR(n, L, rotation_angle=0.0, potential=potential, l=1)
    traj = track_pole(dvrsystem, phis)
    vals = [e for _, e in traj if e is not None]
    if len(vals) < 4:
        continue
    # stabilization check: pole barely moves over the last several phi steps
    tail = np.array(vals[-5:])
    drift = np.max(np.abs(tail - tail.mean()))
    if drift < 1e-3:
        E = tail[-1]
        print(f"V0={v:.4f}: stabilized E = {E.real:.5f} {E.imag:+.5f}i, drift={drift:.2e}")